In [ ]:
"""Loads the three trained models and scores the ISIC 2018 test images. Compares results with the human readers."""

import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from torchvision import models, transforms

from lesion_dataset import LesionDataset

SPLITS = Path("splits")
RUNS = Path("runs")
SEEDS = [1, 2, 3]
BATCH_SIZE = 32
NUM_WORKERS = 4
IMAGE_SIZE = 224
MALIGNANT_TYPES = ["mel", "bcc", "akiec"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# results from Tschandl et al. (2019)
DERM_SENSITIVITY = 0.76 # mean individual reader
DERM_SPECIFICITY = 0.78 # mean individual reader
DERM_VOTE_AUROC = 0.958# all readers combined by majority vote

In [ ]:
# load the test set
test_df = pd.read_csv(SPLITS / "test.csv")
n_pos = int(test_df["label"].sum())
print(f"Test: {len(test_df)} images ({n_pos} malignant, {len(test_df) - n_pos} benign)")
print(test_df["dx"].value_counts()) # print num images for each diagnosis

Test: 1511 images (307 malignant, 1204 benign)
dx
nv       908
bkl      217
mel      171
bcc       93
df        44
akiec     43
vasc      35
Name: count, dtype: int64


In [ ]:
# images stay the same
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# provides the batches for the training iterations
test_loader = DataLoader(LesionDataset(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def load_model(path, device):
    """Rebuild the trained model from saved file. Return (model, cutoff)."""
    checkpoint = torch.load(path, map_location="cpu") # load on cpu first
    model = models.resnet50(weights=None) # empty ResNet-50
    model.fc = nn.Linear(model.fc.in_features, 1) # 1-output last layer 
    model.load_state_dict(checkpoint["model"]) # fill in trained weights
    model.to(device)
    model.eval()
    return model, checkpoint["cutoff"] # cutoff chosen on validation


# check loading works
device = pick_device()
model, cutoff = load_model(RUNS / "seed1" / "best_model.pt", device)
print("Device:", device, "| seed 1 cutoff:", round(cutoff, 3))

Device: mps | seed 1 cutoff: 0.078


In [12]:
def predict(model, loader, device):
    """Scores every image in the loader. Return the probabilities."""
    model.eval()
    all_probs = []

    with torch.no_grad():
        for images, _ in loader:
            outputs = model(images.to(device)).squeeze(1)
            all_probs.append(torch.sigmoid(outputs).cpu())

    return torch.cat(all_probs).numpy()


def sens_spec(labels, probs, cutoff):
    """Sensitivity and specificity when scores at or above the cutoff are called malignant. Return a value between 0 and 1 for each."""
    predicted_malignant = probs >= cutoff
    is_malignant = labels == 1
    is_benign = labels == 0
    sensitivity = (predicted_malignant & is_malignant).sum() / is_malignant.sum()
    specificity = (~predicted_malignant & is_benign).sum() / is_benign.sum()
    return float(sensitivity), float(specificity)


def evaluate_seed(seed):
    """Score the test set with one seed's model at its saved cutoff. Returns one results row (a dictionary) and saves each image's score."""
    model, cutoff = load_model(RUNS / f"seed{seed}" / "best_model.pt", device) # load this seed's model
    probs = predict(model, test_loader, device) # score test images
    labels = test_df["label"].to_numpy()
    sensitivity, specificity = sens_spec(labels, probs, cutoff) # measure sensitivity and specificity using cutoff

    # find the fraction the model flagged as malignant for each malignant type
    row = {"seed": seed, "cutoff": cutoff, "auroc": float(roc_auc_score(labels, probs)), "sensitivity": sensitivity, "specificity": specificity}
    for dx in MALIGNANT_TYPES:
        mask = (test_df["dx"] == dx).to_numpy() # true for rows with current diagnosis dx
        row[f"sens_{dx}"] = float((probs[mask] >= cutoff).mean())

    # save each image's score
    preds = test_df[["image_id", "dx", "label"]].copy()
    preds["prob_malignant"] = probs
    preds.to_csv(RUNS / f"seed{seed}" / "test_predictions.csv", index=False)
    return row

In [ ]:
rows = [evaluate_seed(seed) for seed in SEEDS] # stores one results row per seed
results = pd.DataFrame(rows)
results.to_csv(RUNS / "test_results.csv", index=False)
print(results.round(3).to_string(index=False))

# get mean and standard deviation of each metric
summary = results.drop(columns=["seed", "cutoff"]).agg(["mean", "std"]).T
print("\nMean ± SD across seeds:")
for metric, r in summary.iterrows():
    print(f"  {metric:15s} {r['mean']:.3f} ± {r['std']:.3f}")

# compare with human readers
mean_sens = results["sensitivity"].mean()
mean_spec = results["specificity"].mean()
print(f"\nModel:          sensitivity {mean_sens:.3f} at specificity {mean_spec:.3f}")
print(f"Dermatologists: sensitivity {DERM_SENSITIVITY:.3f} at specificity {DERM_SPECIFICITY:.3f}")
print(f"Difference in sensitivity: {(mean_sens - DERM_SENSITIVITY) * 100:+.1f} points")
print(f"Model AUROC {results['auroc'].mean():.3f} vs dermatologist majority vote {DERM_VOTE_AUROC:.3f}")

 seed  cutoff  auroc  sensitivity  specificity  sens_mel  sens_bcc  sens_akiec
    1   0.078  0.932        0.928        0.733     0.889     0.989       0.953
    2   0.043  0.926        0.899        0.743     0.842     0.957       1.000
    3   0.106  0.929        0.932        0.733     0.889     0.989       0.977

Mean ± SD across seeds:
  auroc           0.929 ± 0.003
  sensitivity     0.920 ± 0.018
  specificity     0.737 ± 0.006
  sens_mel        0.873 ± 0.027
  sens_bcc        0.978 ± 0.019
  sens_akiec      0.977 ± 0.023

Model:          sensitivity 0.920 at specificity 0.737
Dermatologists: sensitivity 0.760 at specificity 0.780
Difference in sensitivity: +16.0 points
Model AUROC 0.929 vs dermatologist majority vote 0.958


In [ ]:
"""Finds the cutoff that gives the readers' 78% specificity. Print each seed's sensitivity at that cutoff."""
from sklearn.metrics import roc_curve

matched = []
for seed in SEEDS:
    # use the saved scores
    preds = pd.read_csv(RUNS / f"seed{seed}" / "test_predictions.csv")
    fpr, tpr, thresholds = roc_curve(preds["label"], preds["prob_malignant"]) # every candidate cutoff
    specificity = 1 - fpr
    # lowest cutoff that gives at least 78% specificity
    i = np.where(specificity >= DERM_SPECIFICITY)[0][-1]
    matched.append({"seed": seed, "cutoff": thresholds[i], "specificity": specificity[i], "sensitivity": tpr[i]})

matched = pd.DataFrame(matched)
print(matched.round(3).to_string(index=False))
print(f"\nAt matched specificity: sensitivity {matched['sensitivity'].mean():.3f} ± "
      f"{matched['sensitivity'].std():.3f} (dermatologists {DERM_SENSITIVITY:.2f})")

 seed  cutoff  specificity  sensitivity
    1   0.161        0.782        0.922
    2   0.083        0.797        0.896
    3   0.231        0.782        0.902

At matched specificity: sensitivity 0.907 ± 0.014 (dermatologists 0.76)
